# 05 - Rental and directly touching neighbor averages
Uses backend assessor/realtor exports in `raw_data/registvr_neighborhood_*.csv`, the complete license inventory, the full CAD matching audit from notebook 03, parcel digest and saved commercial exclusions. Reads the source snapshot offline by default. No credentials are stored in this notebook.

For each eligible rental parcel, select directly touching residential neighbors, deduplicate selected calls inside that neighborhood, and divide by `1 + neighbor_count`. Group plots show the arithmetic mean of those per-rental ratios by licensed occupancy. This differs from pooling calls/parcel memberships across a capacity group. Zero-call parcels and no-neighbor rentals remain included.

Focal types use explicit single-family/duplex evidence, not bedroom counts or building cards as a substitute for dwelling counts. Bedrooms > 0 screen residential parcels; missing bedrooms are excluded per user instruction. Unknown types stay in review. Neighbors are any positive-bedroom, nonexcluded residential parcels. Duplex-with-carriage-house requires explicit supporting evidence and may have no qualifying records. Read the generated page for exact rules and audits.

Calls are parking, noise, loud party and nonweapon nuisance. All initiation types are included in category plots, with a separate citizen-only overall series. Not verified 911 origin or guest involvement. The 2024 CAD period predates current license and building snapshots.

In [1]:
from pathlib import Path
import sys, runpy
from IPython.display import display, Image
ROOT=Path.cwd().resolve()
if not (ROOT/'raw_data').is_dir(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
# Recompute only when the digest/license/exclusion source has changed.
REFRESH_BOUNDARIES=False
# Requires network, verified SSH known-hosts and an authorized key outside this repository.
REFRESH_BACKEND=False
if REFRESH_BOUNDARIES:
    runpy.run_module('processing.prepare_neighborhoods',run_name='__main__')
if REFRESH_BACKEND:
    runpy.run_module('processing.fetch_neighborhood_backend',run_name='__main__')
from processing.neighborhood_occupancy_analysis import run
summary=run(ROOT)
display(summary)

{'focal_rental_parcels': 253,
 'focal_types': {'single_family': 249, 'duplex': 4},
 'residential_neighbor_links': 482,
 'zero_neighbor_rentals': 9,
 'unique_calls_in_included_neighborhoods': 43,
 'call_memberships': 52,
 'conflicting_call_numbers': 0,
 'formula': 'For each focal parcel: unique selected neighborhood calls / (1 + eligible touching neighbors); plots show arithmetic mean of these ratios within occupancy groups.',
 'source_sha256': {'raw_data/registvr_neighborhood_assessor.csv': '4d08137c6cd0c55b8c08ff9da7c782562117cc317b1238c395ebeb57b119580f',
  'raw_data/registvr_neighborhood_realtor.csv': '5a7092c7111c13d1762989d19b37085079f730fbbfadc15213169fd00d83ad10',
  'excluded_properties.csv': '2d0ec122bb34d7d123fa5e4206df26157ec3fc21d7710abcfb1524e04b8f7421',
  'raw_data/CAD-log.xlsx': '2fab32a09b654fc49bef54418195c235c875221328d0c233bb793c5b278ce5d1',
  'raw_data/Parcel_Digest_2025.geojson': '5b0d8d45109801bfea0741fbd4b717da8c87a0e83e222c9ec745e881ca9d6883'}}

In [2]:
import pandas as pd
metrics=pd.read_csv(ROOT/'neighborhood_occupancy_plots/neighborhood_metrics.csv')
assert metrics.property_count.eq(metrics.neighbor_count+1).all()
assert metrics.focal_pin.is_unique
assert metrics.selected_calls.eq(metrics[['Parking complaints','Noise complaints','Loud party','Nuisance']].sum(axis=1)).all()
print('Neighborhood denominator and call-category checks passed.')
print('Individual citizen-only plots: plots/README.md')

Neighborhood denominator and call-category checks passed.
